# Day 2 Lab - Windows-Style Log Generator & Elastic Cloud Uploader
### AIOps Fundamentals Training | Microland

---

## What This Notebook Does

This notebook simulates the operational log data an IT team collects in a real environment.  
It generates **1,000 Windows-style log entries** spanning the last 4 hours, with **20 injected anomalies**
hidden inside the normal traffic - exactly like a brute-force attempt buried in a busy Windows event log.

Once generated, all data is uploaded directly to your **Elastic Cloud** deployment.

---

## Learning Objectives
- Understand what normalised ECS-aligned operational log data looks like
- See how 'normal' log patterns differ from anomalous patterns
- Practice navigating event data in Kibana Discover before ML is applied

---

## Before You Start
1. You have a running **Elastic Cloud** deployment (created in the Day 1 lab)
2. You have your **Cloud ID** and **API Key** ready (from the Elastic Cloud console)
3. You are running this in **Google Colab** (no local install needed)

> **No Python knowledge required.** Run each cell in order using the Play button or Shift+Enter.


## Step 1 - Install Required Libraries
This installs the Elasticsearch Python client. Takes about 15 seconds.


In [ ]:
!pip install elasticsearch --quiet
print('Libraries installed successfully.')


## Step 2 - Enter Your Elastic Cloud Connection Details

**Where to find these values:**
1. Go to cloud.elastic.co and open your deployment
2. **Cloud ID**: shown on the deployment overview page - click Copy
3. **API Key**: go to Stack Management > API Keys > Create API Key - copy the encoded key

> **Important:** Never share your API key. Treat it like a password.


In [ ]:
# -----------------------------------------------------------
# CONFIGURATION - Fill in your own values here
# -----------------------------------------------------------

CLOUD_ID   = 'YOUR_CLOUD_ID_HERE'   # Paste your Elastic Cloud ID
API_KEY    = 'YOUR_API_KEY_HERE'    # Paste your Elastic API Key
INDEX_NAME = 'aiops-lab-day2'       # Index to create in Elasticsearch

print(f'Configuration set.')
print(f'   Target index : {INDEX_NAME}')


## Step 3 - Connect to Elastic Cloud
This cell tests that your credentials are correct before uploading any data.


In [ ]:
from elasticsearch import Elasticsearch

es = Elasticsearch(
    cloud_id=CLOUD_ID,
    api_key=API_KEY
)

info = es.info()
print(f'Connected to Elasticsearch cluster: {info["cluster_name"]}')
print(f'   Elasticsearch version : {info["version"]["number"]}')


## Step 4 - Create the Index with ECS-Aligned Mapping

We define the field structure (called a **mapping**) before uploading data.  
The field names below mirror the **Elastic Common Schema (ECS)** - the same schema
that Winlogbeat and Filebeat use when shipping real Windows events.

> **AIOps Concept:** Without a consistent schema, ML cannot compare events from different sources.
> The mapping below enforces consistency so our simulated data behaves like real production data.


In [ ]:
# Delete index if it already exists (safe for lab re-runs)
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f'Deleted existing index: {INDEX_NAME}')

mapping = {
    'mappings': {
        'properties': {
            '@timestamp':          {'type': 'date'},
            'event.action':        {'type': 'keyword'},
            'event.category':      {'type': 'keyword'},
            'event.outcome':       {'type': 'keyword'},
            'event.code':          {'type': 'keyword'},
            'event.dataset':       {'type': 'keyword'},
            'host.name':           {'type': 'keyword'},
            'host.os.name':        {'type': 'keyword'},
            'user.name':           {'type': 'keyword'},
            'source.ip':           {'type': 'ip'},
            'source.port':         {'type': 'integer'},
            'message':             {'type': 'text'},
            'log.level':           {'type': 'keyword'},
            'is_anomaly':          {'type': 'boolean'},
            'anomaly_description': {'type': 'text'}
        }
    }
}

es.indices.create(index=INDEX_NAME, body=mapping)
print(f'Index "{INDEX_NAME}" created with ECS-aligned mapping.')


## Step 5 - Generate 1,000 Log Events

This cell builds the dataset:

| Event Type | Count | Description |
|------------|-------|-------------|
| Normal logon events (Event ID 4624) | ~500 | Successful logins from known internal IPs |
| Normal failed logons (Event ID 4625) | ~200 | Occasional failed logins - normal background noise |
| Service events (Event ID 7036) | ~200 | Services starting and stopping - routine activity |
| Process creation (Event ID 4688) | ~80 | New processes created - normal admin activity |
| **Injected anomalies** (Event ID 4625) | **20** | Rapid failed logins from one suspicious external IP |

> **Your challenge in Kibana:** Can you find the 20 anomalous events hidden in the 980 normal ones?


In [ ]:
import random
import json
from datetime import datetime, timedelta, timezone

random.seed(42)  # Fixed seed for reproducibility

# Time range: last 4 hours
now        = datetime.now(timezone.utc)
start_time = now - timedelta(hours=4)

# Reference data
SERVERS      = ['WINTEL-SRV-01', 'WINTEL-SRV-02', 'WINTEL-SRV-03',
                 'DC-PROD-01', 'DC-PROD-02', 'APP-SRV-07', 'FILE-SRV-03']
INTERNAL_IPS = ['10.10.1.10', '10.10.1.11', '10.10.1.25',
                 '192.168.1.50', '192.168.1.51', '192.168.10.20']
DOMAIN_USERS = ['jsmith', 'aparikh', 'nkumar', 'rdesai', 'srao',
                 'admin_it', 'svc_backup', 'svc_monitor']
PROCESSES    = ['svchost.exe', 'lsass.exe', 'explorer.exe', 'powershell.exe',
                 'cmd.exe', 'mmc.exe', 'taskmgr.exe']
SERVICES     = ['Windows Update', 'Print Spooler', 'DHCP Client',
                 'DNS Client', 'Windows Defender', 'Remote Desktop Services']

# Anomaly: 20 rapid failed logins from one suspicious external IP in a 3-minute window
ANOMALY_IP   = '185.220.101.47'   # Known Tor exit-node range
ANOMALY_USER = 'administrator'
anomaly_start = start_time + timedelta(hours=2, minutes=15)
anomaly_end   = anomaly_start + timedelta(minutes=3)


def rand_ts(t0, t1):
    return t0 + timedelta(seconds=random.uniform(0, (t1 - t0).total_seconds()))


def ev_logon_ok(ts):
    u, ip, h = random.choice(DOMAIN_USERS), random.choice(INTERNAL_IPS), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'logged-in',
        'event.category': 'authentication', 'event.outcome': 'success',
        'event.code': '4624', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': u, 'source.ip': ip, 'source.port': random.randint(49152, 65535),
        'log.level': 'information',
        'message': f'An account was successfully logged on. Account: {u}, IP: {ip}',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_logon_fail(ts, ip=None, user=None):
    u, src, h = (user or random.choice(DOMAIN_USERS)), (ip or random.choice(INTERNAL_IPS)), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'authentication_failure',
        'event.category': 'authentication', 'event.outcome': 'failure',
        'event.code': '4625', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': u, 'source.ip': src, 'source.port': random.randint(49152, 65535),
        'log.level': 'warning',
        'message': f'An account failed to log on. Account: {u}, IP: {src}, Reason: Unknown user name or bad password',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_service(ts):
    svc, h, state = random.choice(SERVICES), random.choice(SERVERS), random.choice(['started', 'stopped'])
    return {
        '@timestamp': ts.isoformat(), 'event.action': f'service-{state}',
        'event.category': 'process', 'event.outcome': 'success',
        'event.code': '7036', 'event.dataset': 'system.system',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': 'SYSTEM', 'source.ip': '127.0.0.1', 'source.port': 0,
        'log.level': 'information',
        'message': f'The {svc} service entered the {state} state.',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_process(ts):
    proc, user, h = random.choice(PROCESSES), random.choice(DOMAIN_USERS), random.choice(SERVERS)
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'process_started',
        'event.category': 'process', 'event.outcome': 'success',
        'event.code': '4688', 'event.dataset': 'system.security',
        'host.name': h, 'host.os.name': 'Windows Server 2019',
        'user.name': user, 'source.ip': random.choice(INTERNAL_IPS), 'source.port': 0,
        'log.level': 'information',
        'message': f'A new process was created. Name: {proc}, Creator: {user}',
        'is_anomaly': False, 'anomaly_description': ''
    }


def ev_anomaly(ts):
    return {
        '@timestamp': ts.isoformat(), 'event.action': 'authentication_failure',
        'event.category': 'authentication', 'event.outcome': 'failure',
        'event.code': '4625', 'event.dataset': 'system.security',
        'host.name': 'DC-PROD-01', 'host.os.name': 'Windows Server 2019',
        'user.name': ANOMALY_USER, 'source.ip': ANOMALY_IP,
        'source.port': random.randint(49152, 65535),
        'log.level': 'warning',
        'message': f'An account failed to log on. Account: {ANOMALY_USER}, IP: {ANOMALY_IP}, Reason: Unknown user name or bad password',
        'is_anomaly': True,
        'anomaly_description': 'Brute-force: 20 rapid failed logins from external IP 185.220.101.47 against administrator in a 3-minute window'
    }


# Build dataset
events = []
for _ in range(500): events.append(ev_logon_ok(rand_ts(start_time, now)))
for _ in range(200): events.append(ev_logon_fail(rand_ts(start_time, now)))
for _ in range(200): events.append(ev_service(rand_ts(start_time, now)))
for _ in range(80):  events.append(ev_process(rand_ts(start_time, now)))
for _ in range(20):  events.append(ev_anomaly(rand_ts(anomaly_start, anomaly_end)))

events.sort(key=lambda x: x['@timestamp'])

normal_count  = sum(1 for e in events if not e['is_anomaly'])
anomaly_count = sum(1 for e in events if     e['is_anomaly'])

print(f'Generated {len(events):,} log events')
print(f'   Normal events  : {normal_count:,}')
print(f'   Anomaly events : {anomaly_count:,}  <- hidden in the stream')
print(f'   Time range     : {events[0]["@timestamp"][:19]}  to  {events[-1]["@timestamp"][:19]}')
print(f'   Anomaly window : {anomaly_start.strftime("%H:%M:%S")}  to  {anomaly_end.strftime("%H:%M:%S")} UTC')
print()
print('Sample - most recent event:')
print(json.dumps(events[-1], indent=2))


## Step 6 - Upload to Elastic Cloud

All 1,000 events are uploaded in one efficient batch using the Elasticsearch bulk API.


In [ ]:
from elasticsearch.helpers import bulk

def generate_actions(docs, index):
    for doc in docs:
        yield {'_index': index, '_source': doc}

print(f'Uploading {len(events):,} events to index "{INDEX_NAME}" ...')

success, errors = bulk(
    es,
    generate_actions(events, INDEX_NAME),
    chunk_size=200,
    raise_on_error=False
)

print(f'Upload complete!')
print(f'   Documents indexed : {success:,}')
if errors:
    print(f'   Errors            : {len(errors)}')
    for e in errors[:3]: print(f'      {e}')
else:
    print(f'   Errors            : 0')

es.indices.refresh(index=INDEX_NAME)
print('Index refreshed - data is now searchable in Kibana.')


## Step 7 - Verify the Upload


In [ ]:
total   = es.count(index=INDEX_NAME)['count']
anomaly = es.count(index=INDEX_NAME, body={'query': {'term': {'is_anomaly': True}}})['count']

agg = es.search(index=INDEX_NAME, body={
    'size': 0,
    'aggs': {'by_action': {'terms': {'field': 'event.action', 'size': 10}}}
})

print(f'Verification summary - index "{INDEX_NAME}":')
print(f'   Total documents : {total:,}')
print(f'   Anomaly events  : {anomaly:,}  <- try to find these in Kibana!')
print()
print('Event breakdown by action:')
for b in agg['aggregations']['by_action']['buckets']:
    print(f'   {b["key"]:35s}  {b["doc_count"]:>5,} events')


## Step 8 - Explore Your Data in Kibana

### 8a - Open Kibana Discover
1. Go to your Elastic Cloud deployment and click **Launch Kibana**
2. In the left navigation menu, click **Discover**
3. If prompted to create a data view:
   - Index pattern: `aiops-lab-day2`
   - Timestamp field: `@timestamp`
   - Click **Save data view to Kibana**

---

### 8b - KQL Queries to Try in Kibana

Copy and paste these into the Kibana search bar:

```
# Show only failed logons
event.action : "authentication_failure"

# Show only the injected anomalies
is_anomaly : true

# Show events from the suspicious IP
source.ip : "185.220.101.47"

# Combine filters
event.action : "authentication_failure" AND source.ip : "185.220.101.47"
```

---

### 8c - Build the Dashboard (Lab Task)

Go to **Dashboards > Create dashboard > Add visualisation** and create:

| Panel | Type | Field |
|-------|------|-------|
| Top event actions | Bar chart | `event.action` |
| Events over time | Line chart | `@timestamp` |
| Source IP table | Data table | `source.ip` |

---

### Challenge Question

> The 20 anomalous events from IP `185.220.101.47` are hidden among 200 normal failed logons.
> Can you spot them visually in your bar chart or timeline?
> **This is exactly why we need ML** - human eyes cannot reliably detect low-volume,
> time-clustered anomalies in dashboards full of noise.

---

### Lab Complete
Move to **Day 3** where you will build an ML job to detect anomalies like this automatically.


## Optional - Preview the 20 Injected Anomaly Events Locally
Run this cell to see exactly what the brute-force events look like before searching for them in Kibana.


In [ ]:
import pandas as pd

anom_df = pd.DataFrame([e for e in events if e['is_anomaly']])[
    ['@timestamp', 'event.action', 'event.outcome', 'host.name', 'user.name', 'source.ip']
]
anom_df['@timestamp'] = pd.to_datetime(anom_df['@timestamp']).dt.strftime('%H:%M:%S')
anom_df = anom_df.rename(columns={'@timestamp': 'Time (UTC)'})

print('The 20 injected anomaly events (brute-force window):')
print(anom_df.to_string(index=False))
